# HCZ DAS View Tutorial and User Manual

HCZ DAS View is a **DAS Viewer / DAS Analysis** package. It focuses on reading DAS data, displaying metadata, visualizing time-channel data, plotting waveform/spectrum/PSD/spectrogram/FK views, running common preprocessing and analysis, and supporting a maintainable GUI workflow.

It is not a surface-wave inversion, MASW, F-J, dispersion-picking, earthquake-location, or source-location package. Event outputs in this manual are **event candidates** for DAS data inspection, not location or interpretation results.

## 1. Software Role: DAS Viewer / DAS Analysis

The package is intended for practical DAS data inspection and analysis:

- file reading and metadata summary;
- waterfall and waveform display;
- preprocessing and filtering;
- spectrum, PSD, spectrogram, statistics, spectral attributes, FK visualization;
- envelope, STA/LTA, and event candidate tables;
- CLI and GUI workflows.

## 2. DAS Data Shape: time x channel

All internal arrays follow:

```text
data.shape == (n_samples, n_channels)
```

Axis convention:

- `axis=0`: time samples;
- `axis=1`: channel / space axis.

Readers convert external file orientation into this convention before data reaches analysis functions.

In [ ]:
import numpy as np

sample_rate_hz = 1000.0
t = np.arange(1024) / sample_rate_hz
synthetic = np.column_stack([
    np.sin(2 * np.pi * 10.0 * t),
    0.5 * np.sin(2 * np.pi * 30.0 * t),
])
synthetic.shape

## 3. Reading Data and Metadata

Use reader-independent services for application code. A typical file workflow starts from metadata and bounded selections rather than full-file reads.

```python
from das_view.io.data_service import create_preview, read_selection

preview = create_preview("sample.h5", max_samples=2000, max_channels=512)
selection = read_selection("sample.h5", time_slice=slice(0, 4096), channel_slice=slice(0, 128))
metadata = selection.das_data.metadata
```

Use local paths in your own session; do not put private data paths in shared notebooks or documentation.

## 4. Waterfall and Waveform Views

A waterfall view shows the DAS matrix as a time-channel image. A waveform view shows one or more channels as traces. These are inspection tools for signal quality, clipping, time windows, active channels, and obvious events.

CLI examples:

```bash
python examples/validate_file.py sample.h5
python examples/plot_waveform.py sample.h5 --channel 0 --output waveform.png
```

## 5. Common Preprocessing

Stable preprocessing operations include:

- `demean`: subtract mean along a selected axis;
- `detrend`: remove linear trend;
- `taper`: reduce edge effects before spectral analysis;
- `normalize`: max-absolute, min-max, or standard-score scaling;
- `clip`: limit extreme amplitudes.

Preprocessing should be documented in analysis outputs because it changes measured attributes.

In [ ]:
from das_view.processing.preprocess import demean, normalize

processed = demean(synthetic, axis=0)
processed = normalize(processed, method="maxabs")
processed.shape

## 6. Common Filtering

Filtering tools include lowpass, highpass, bandpass, bandstop/notch-style filters through SciPy-backed implementations. Always check sample rate and Nyquist frequency before selecting cutoff frequencies.

```python
from das_view.processing.filters import bandpass

filtered = bandpass(data, sample_rate_hz=1000.0, low_hz=5.0, high_hz=80.0, axis=0)
```

## 7. Spectrum, PSD, and Spectrogram

Spectrum tools describe frequency content.

- Amplitude spectrum estimates frequency-domain amplitude.
- PSD estimates power per frequency using periodogram or Welch methods.
- Spectrogram shows time-varying frequency content.

Use bounded selections for large DAS files.

```bash
python examples/spectrum_file.py sample.h5 --channel 0
python examples/spectrum_file.py sample.h5 --channel 0 --psd welch
python examples/spectrum_file.py sample.h5 --channel 0 --spectrogram
```

## 8. Statistical Attributes

Basic statistics summarize amplitude behavior globally, time-wise, or channel-wise.

Key formulas:

```text
RMS = sqrt(mean(x^2))
Energy = sum(x^2)
Peak-to-peak = max(x) - min(x)
```

Supported attributes include mean, standard deviation, RMS, min, max, median, percentiles, absolute mean, peak-to-peak, energy, and finite/NaN/Inf summaries.

```bash
python examples/statistics_file.py sample.h5
python examples/statistics_file.py sample.h5 --axis time
python examples/statistics_file.py sample.h5 --axis channel
```

## 9. Spectral Attributes

Spectral attributes describe frequency-domain behavior for each channel or an averaged selection.

Key formulas:

```text
Band energy = sum P(f), f in [f1, f2]
Spectral centroid = sum(f P(f)) / sum(P(f))
Spectral bandwidth = sqrt(sum((f - centroid)^2 P(f)) / sum(P(f)))
```

Additional attributes include dominant frequency, peak spectral power, rolloff frequency, and band energy ratio.

```bash
python examples/spectral_attributes_file.py sample.h5 --bands 1 5 5 20
python examples/spectral_attributes_file.py sample.h5 --attributes --frequency-range 1 80
```

## 10. FK Visualization and FK-domain Smoke Filtering

FK tools inspect 2-D DAS wavefield content in frequency-wavenumber space. Apparent velocity is related to frequency and wavenumber:

```text
FK apparent velocity = |f / k|
```

FK-domain smoke filtering can suppress or preserve broad velocity ranges for DAS 2-D wavefield inspection. It is not a surface-wave imaging, inversion, MASW, F-J, or dispersion-picking workflow.

```bash
python examples/fk_file.py sample.h5 --time-start 0 --time-stop 4096 --channel-start 0 --channel-stop 512
python examples/fk_filter_file.py sample.h5 --time-start 0 --time-stop 4096 --channel-start 0 --channel-stop 512 --vmin 300 --vmax 3000
```

## 11. Envelope, STA/LTA, and Event Candidates

Event candidate tools highlight amplitude or energy changes in DAS data. They produce candidate intervals and channel indices, not source locations.

Key formulas:

```text
Envelope = |Hilbert(x)|
Short-window energy = moving_sum(x^2, STA window)
Long-window energy = moving_sum(x^2, LTA window)
STA/LTA = short_window_energy / long_window_energy
```

A threshold crossing creates an event candidate with start/end sample, duration, channel range, peak sample/channel, peak value, mean value, and score. Interpret candidates with context and visual review.

In [ ]:
from das_view.analysis.events import amplitude_envelope, sta_lta_ratio, detect_threshold_events

feature = amplitude_envelope(synthetic).values
ratio = sta_lta_ratio(synthetic, sta_samples=20, lta_samples=100).ratio
candidates = detect_threshold_events(feature, threshold=0.8, max_events=5).candidates
len(candidates)

## 12. CLI User Guide

Common CLI workflows:

```bash
python examples/validate_file.py sample.h5
python examples/statistics_file.py sample.h5 --output stats.json
python examples/spectral_attributes_file.py sample.h5 --bands 1 5 5 20 --output bands.json
python examples/event_detection_file.py sample.h5 --method stalta --sta 50 --lta 500 --trigger-on 3.0
python examples/event_detection_file.py sample.h5 --method envelope --threshold 0.8 --output events.csv
```

For large files, use `--time-start`, `--time-stop`, `--channel-start`, and `--channel-stop` to keep reads bounded.

## 13. GUI User Guide

Typical GUI workflow:

1. Launch the GUI with `python examples/run_gui.py` or the installed console script.
2. Use **Open File** to select a supported DAS file.
3. Check metadata first: sample count, channel count, sample rate, spacing, and source format.
4. Use Waterfall for time-channel inspection.
5. Use Waveform for selected channels.
6. Use Spectrum for amplitude spectrum, Welch PSD, or spectrogram views.
7. Use FK only when time and channel spacing metadata are suitable.

Statistics, spectral attributes, and event candidates are currently available through analysis functions and CLI examples; GUI integration belongs to a later stable workflow.

## 14. Current Limits and Interpretation Boundaries

Use this package as a DAS viewer and analysis toolkit. Current boundaries:

- Event candidate tables are screening aids, not earthquake locations or source-location results.
- FK visualization and FK-domain filtering are wavefield inspection tools, not specialized inversion workflows.
- Results depend on preprocessing, selected time/channel window, sample rate, channel spacing, thresholds, and noise conditions.
- Always inspect candidates visually and compare before/after preprocessing when analysis affects decisions.